# CodeGuard — CyberSecEval4 results visualization

Charts and tables for **instruct** and **autocomplete** on three base models:
`gemma3:4b`, `gpt-oss:20b`, and `gpt-oss:120b` (baseline vs CodeGuard).

All metrics come from `instruct_stat.json` and `autocomplete_stat.json` under
`eval/<benchmark>/results/`. No estimated or placeholder values.

Run from `eval/visualization/`:
```bash
pip install pandas plotly
jupyter notebook codeguard_results_visualization.ipynb
```


In [6]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import plotly.graph_objects as go

EVAL_DIR = Path(".").resolve()  # eval/visualization
EVAL_BASE = EVAL_DIR.parent
INSTRUCT_RESULTS = EVAL_BASE / "instruct" / "results"
AUTOCOMPLETE_RESULTS = EVAL_BASE / "autocomplete" / "results"
INSTRUCT_STAT = INSTRUCT_RESULTS / "instruct_stat.json"
AUTOCOMPLETE_STAT = AUTOCOMPLETE_RESULTS / "autocomplete_stat.json"

METRICS = ["pass_rate", "vulnerable_percentage", "bleu"]
METRIC_LABELS = {
    "pass_rate": "Pass rate (%)",
    "vulnerable_percentage": "Vulnerable (%)",
    "bleu": "BLEU",
}

COLOR_BASELINE = "#4C72B0"
COLOR_CODEGUARD = "#C41E3A"

MODEL_SIZES_GB = {
    "gemma3:4b": 3.3,
    "gpt-oss:20b": 14.0,
    "gpt-oss:120b": 65.0,
}

SIZE_BASE_MODELS = list(MODEL_SIZES_GB.keys())

# Stat-file keys -> canonical base model id
BASE_KEY_ALIASES = {
    "gemma3:4b": "gemma3:4b",
    "gpt-oss:20b": "gpt-oss:20b",
    "gpt-oss:20b-cloud": "gpt-oss:20b",
    "gpt-oss:120b": "gpt-oss:120b",
    "gpt-oss:120b-cloud": "gpt-oss:120b",
}

CODEGUARD_KEY_ALIASES = {
    "codeguard-gemma3:4b": "gemma3:4b",
    "codeguard-gpt-oss:20b": "gpt-oss:20b",
    "codeguard-gpt-oss:20b-cloud": "gpt-oss:20b",
    "codeguard-gpt-oss:120b": "gpt-oss:120b",
}

PREFERRED_CODEGUARD_KEY = {
    "gemma3:4b": "codeguard-gemma3:4b",
    "gpt-oss:20b": "codeguard-gpt-oss:20b",
    "gpt-oss:120b": "codeguard-gpt-oss:120b",
}

RESPONSE_FILE_MAP: dict[tuple[str, str, bool], Path] = {
    ("instruct", "gemma3:4b", False): INSTRUCT_RESULTS / "instruct_responses_gemma3:4b.json",
    ("instruct", "gpt-oss:20b", False): INSTRUCT_RESULTS / "instruct_responses_gpt-oss:20b-cloud.json",
    ("instruct", "gpt-oss:120b", False): INSTRUCT_RESULTS / "instruct_responses_gpt-oss:120b-cloud.json",
    ("instruct", "gemma3:4b", True): INSTRUCT_RESULTS / "instruct_responses_codeguard_gemma3:4b.json",
    ("instruct", "gpt-oss:20b", True): INSTRUCT_RESULTS / "instruct_responses_codeguard_gpt-oss:20b-cloud.json",
    ("instruct", "gpt-oss:120b", True): INSTRUCT_RESULTS / "instruct_responses_codeguard_gpt-oss:120b.json",
    ("autocomplete", "gemma3:4b", False): AUTOCOMPLETE_RESULTS / "autocomplete_responses_gemma3:4b.json",
    ("autocomplete", "gpt-oss:20b", False): AUTOCOMPLETE_RESULTS / "autocomplete_responses_gpt-oss:20b-cloud.json",
    ("autocomplete", "gpt-oss:120b", False): AUTOCOMPLETE_RESULTS / "autocomplete_responses_gpt-oss:120b-cloud.json",
    ("autocomplete", "gemma3:4b", True): AUTOCOMPLETE_RESULTS / "autocomplete_responses_codeguard_gemma3:4b.json",
    ("autocomplete", "gpt-oss:20b", True): AUTOCOMPLETE_RESULTS / "autocomplete_responses_codeguard_gpt-oss:20b-cloud.json",
    ("autocomplete", "gpt-oss:120b", True): AUTOCOMPLETE_RESULTS / "autocomplete_responses_codeguard_gpt-oss:120b.json",
}


In [7]:
def load_stat(path: Path) -> dict[str, Any]:
    with path.open() as f:
        return json.load(f)


def stat_rows(stat: dict[str, Any], benchmark: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for model_key, langs in stat.items():
        py = langs.get("python", {})
        base = canonical_base(model_key) or canonical_from_codeguard(model_key)
        if base not in SIZE_BASE_MODELS:
            continue
        is_codeguard = model_key.startswith("codeguard-")
        if is_codeguard and model_key != PREFERRED_CODEGUARD_KEY.get(base):
            continue
        rows.append(
            {
                "benchmark": benchmark,
                "base_model": base,
                "model_key": model_key,
                "codeguard": is_codeguard,
                "display_name": model_key,
                **{m: py.get(m) for m in METRICS},
                "total_count": py.get("total_count"),
            }
        )
    return rows


def canonical_base(model_key: str) -> str | None:
    return BASE_KEY_ALIASES.get(model_key)


def canonical_from_codeguard(model_key: str) -> str | None:
    return CODEGUARD_KEY_ALIASES.get(model_key)


def get_baseline_metric(base_model: str, stat: dict[str, Any]) -> dict[str, Any] | None:
    for key, base in BASE_KEY_ALIASES.items():
        if base == base_model and key in stat:
            return stat[key]["python"]
    return None


def get_codeguard_metric(base_model: str, stat: dict[str, Any]) -> dict[str, Any] | None:
    preferred = PREFERRED_CODEGUARD_KEY.get(base_model)
    if preferred and preferred in stat:
        return stat[preferred]["python"]
    return None


def three_models_df() -> pd.DataFrame:
    rows = stat_rows(load_stat(INSTRUCT_STAT), "instruct")
    rows += stat_rows(load_stat(AUTOCOMPLETE_STAT), "autocomplete")
    return pd.DataFrame(rows).sort_values(["benchmark", "base_model", "codeguard"])


def size_comparison_df() -> pd.DataFrame:
    instruct_stat = load_stat(INSTRUCT_STAT)
    autocomplete_stat = load_stat(AUTOCOMPLETE_STAT)
    rows: list[dict[str, Any]] = []

    for benchmark, stat in [("instruct", instruct_stat), ("autocomplete", autocomplete_stat)]:
        for base_model in SIZE_BASE_MODELS:
            baseline = get_baseline_metric(base_model, stat)
            guarded = get_codeguard_metric(base_model, stat)
            if baseline:
                rows.append(
                    {
                        "benchmark": benchmark,
                        "base_model": base_model,
                        "size_gb": MODEL_SIZES_GB[base_model],
                        "variant": "baseline",
                        "codeguard": False,
                        **{m: baseline[m] for m in METRICS},
                    }
                )
            if guarded:
                rows.append(
                    {
                        "benchmark": benchmark,
                        "base_model": base_model,
                        "size_gb": MODEL_SIZES_GB[base_model],
                        "variant": "codeguard",
                        "codeguard": True,
                        **{m: guarded[m] for m in METRICS},
                    }
                )

    return pd.DataFrame(rows).sort_values(["benchmark", "size_gb", "codeguard"])


def mean_response_chars(benchmark: str, base_model: str, codeguard: bool) -> float | None:
    path = RESPONSE_FILE_MAP.get((benchmark, base_model, codeguard))
    if path is None or not path.exists():
        return None
    with path.open() as f:
        data = json.load(f)
    lengths = [len(item.get("response", "")) for item in data if item.get("response")]
    return sum(lengths) / len(lengths) if lengths else None


def output_length_df() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for benchmark in ("instruct", "autocomplete"):
        for base_model in SIZE_BASE_MODELS:
            for codeguard in (False, True):
                mean_chars = mean_response_chars(benchmark, base_model, codeguard)
                if mean_chars is None:
                    continue
                rows.append(
                    {
                        "benchmark": benchmark,
                        "base_model": base_model,
                        "size_gb": MODEL_SIZES_GB[base_model],
                        "codeguard": codeguard,
                        "mean_response_chars": round(mean_chars, 1),
                    }
                )
    return pd.DataFrame(rows).sort_values(["benchmark", "size_gb", "codeguard"])


def _model_label(row: pd.Series, codeguard: bool) -> str:
    name = str(row["base_model"])
    return f"{name} (+CG)" if codeguard else name


def _add_series(
    fig: go.Figure,
    part: pd.DataFrame,
    x_col: str,
    y_col: str,
    *,
    name: str,
    color: str,
    codeguard: bool,
    marker_symbol: str = "circle",
    showlegend: bool = True,
) -> None:
    if part.empty:
        return
    part = part.sort_values(x_col)
    fig.add_trace(
        go.Scatter(
            x=part[x_col],
            y=part[y_col],
            mode="lines+markers",
            name=name,
            legendgroup=name,
            showlegend=showlegend,
            line=dict(color=color, width=2),
            marker=dict(symbol=marker_symbol, size=11, color=color, line=dict(width=1, color="white")),
            hovertemplate=(
                f"{name}<br>%{{customdata}}<br>"
                + "size=%{x:.1f} GB<br>"
                + "value=%{y:.2f}<extra></extra>"
            ),
            customdata=[_model_label(r, codeguard) for _, r in part.iterrows()],
        )
    )
    for _, r in part.iterrows():
        fig.add_annotation(
            x=r[x_col],
            y=r[y_col],
            text=_model_label(r, codeguard),
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            yshift=14,
            font=dict(size=11, color=color),
            bgcolor="rgba(255,255,255,0.85)",
            borderpad=2,
        )


def _pad_y_axis(fig: go.Figure, ratio: float = 0.12) -> None:
    ys = []
    for trace in fig.data:
        if trace.y is not None:
            ys.extend(trace.y)
    if not ys:
        return
    y_min, y_max = min(ys), max(ys)
    span = y_max - y_min or abs(y_max) * 0.1 or 1.0
    fig.update_yaxes(range=[y_min - span * ratio, y_max + span * ratio])


def _finalize_figure(
    fig: go.Figure,
    *,
    title: str,
    y_title: str,
    log_y: bool = False,
) -> None:
    if not log_y:
        _pad_y_axis(fig)
    fig.update_xaxes(
        type="linear",
        title_text="Model size (GB)",
        showgrid=True,
        automargin=True,
    )
    fig.update_yaxes(
        type="log" if log_y else "linear",
        title_text=y_title,
        showgrid=True,
        automargin=True,
    )
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", y=0.97),
        template="plotly_white",
        height=480,
        width=620,
        margin=dict(t=70, b=90, l=70, r=30),
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.22,
            x=0.5,
            xanchor="center",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cccccc",
            borderwidth=1,
        ),
    )
    fig.show()


## 1. Metrics table (three models)

In [8]:
metrics_df = three_models_df()
display_cols = ["benchmark", "base_model", "display_name", "codeguard", *METRICS, "total_count"]
metrics_df[display_cols]

,benchmark,base_model,display_name,codeguard,pass_rate,vulnerable_percentage,bleu,total_count
6,autocomplete,gemma3:4b,gemma3:4b,False,71.225071,28.774929,11.728480,351
9,autocomplete,gemma3:4b,codeguard-gemma3:4b,True,94.871795,5.128205,3.615465,351
7,autocomplete,gpt-oss:120b,gpt-oss:120b,False,64.957265,35.042735,13.779549,351
10,autocomplete,gpt-oss:120b,codeguard-gpt-oss:120b,True,96.011396,3.988604,3.113571,351
8,autocomplete,gpt-oss:20b,gpt-oss:20b,False,62.962963,37.037037,14.897818,351
11,autocomplete,gpt-oss:20b,codeguard-gpt-oss:20b,True,96.011396,3.988604,3.721848,351
0,instruct,gemma3:4b,gemma3:4b,False,78.368794,21.631206,6.796226,282
1,instruct,gemma3:4b,codeguard-gemma3:4b,True,93.971631,6.028369,2.696501,282
3,instruct,gpt-oss:120b,gpt-oss:120b,False,82.269504,17.730496,6.082977,282
2,instruct,gpt-oss:120b,codeguard-gpt-oss:120b,True,92.198582,7.801418,2.780533,282


## 2. Baseline-only table

In [9]:
size_df = size_comparison_df()
baseline_only = size_df[~size_df["codeguard"]].copy()
baseline_only[["benchmark", "base_model", "size_gb", *METRICS]].sort_values(["benchmark", "size_gb"])

,benchmark,base_model,size_gb,pass_rate,vulnerable_percentage,bleu
6,autocomplete,gemma3:4b,3.3,71.225071,28.774929,11.728480
8,autocomplete,gpt-oss:20b,14.0,62.962963,37.037037,14.897818
10,autocomplete,gpt-oss:120b,65.0,64.957265,35.042735,13.779549
0,instruct,gemma3:4b,3.3,78.368794,21.631206,6.796226
2,instruct,gpt-oss:20b,14.0,81.914894,18.085106,7.308147
4,instruct,gpt-oss:120b,65.0,82.269504,17.730496,6.082977


## 3. Model size vs metric — baseline vs CodeGuard

One plot per benchmark and metric.


In [10]:
def plot_size_vs_metric(size_df: pd.DataFrame, metric: str) -> None:
    for benchmark in sorted(size_df["benchmark"].unique()):
        sub = size_df[size_df["benchmark"] == benchmark]
        fig = go.Figure()
        for codeguard, label, color, symbol in [
            (False, "Baseline", COLOR_BASELINE, "circle"),
            (True, "CodeGuard", COLOR_CODEGUARD, "square"),
        ]:
            part = sub[sub["codeguard"] == codeguard]
            _add_series(
                fig,
                part,
                "size_gb",
                metric,
                name=label,
                color=color,
                codeguard=codeguard,
                marker_symbol=symbol,
                showlegend=True,
            )
        _finalize_figure(
            fig,
            title=f"{benchmark.capitalize()} — {METRIC_LABELS[metric]} vs model size",
            y_title=METRIC_LABELS[metric],
        )


for metric in METRICS:
    plot_size_vs_metric(size_df, metric)


## 4. Pivot table — baseline vs CodeGuard

In [11]:
pivot_rows = []
for benchmark in ("instruct", "autocomplete"):
    for base_model in SIZE_BASE_MODELS:
        row = {
            "benchmark": benchmark,
            "base_model": base_model,
            "size_gb": MODEL_SIZES_GB[base_model],
        }
        for codeguard, suffix in [(False, "baseline"), (True, "codeguard")]:
            part = size_df[
                (size_df["benchmark"] == benchmark)
                & (size_df["base_model"] == base_model)
                & (size_df["codeguard"] == codeguard)
            ]
            if part.empty:
                continue
            r = part.iloc[0]
            for m in METRICS:
                row[f"{m}_{suffix}"] = r[m]
        pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).sort_values(["benchmark", "size_gb"])
pivot_df

,benchmark,base_model,size_gb,pass_rate_baseline,vulnerable_percentage_baseline,bleu_baseline,pass_rate_codeguard,vulnerable_percentage_codeguard,bleu_codeguard
3,autocomplete,gemma3:4b,3.3,71.225071,28.774929,11.728480,94.871795,5.128205,3.615465
4,autocomplete,gpt-oss:20b,14.0,62.962963,37.037037,14.897818,96.011396,3.988604,3.721848
5,autocomplete,gpt-oss:120b,65.0,64.957265,35.042735,13.779549,96.011396,3.988604,3.113571
0,instruct,gemma3:4b,3.3,78.368794,21.631206,6.796226,93.971631,6.028369,2.696501
1,instruct,gpt-oss:20b,14.0,81.914894,18.085106,7.308147,94.326241,5.673759,3.190692
2,instruct,gpt-oss:120b,65.0,82.269504,17.730496,6.082977,92.198582,7.801418,2.780533


## 5. Mean output length (characters per response)

Computed from saved response JSON files.

In [12]:
length_df = output_length_df()
length_df

,benchmark,base_model,size_gb,codeguard,mean_response_chars
6,autocomplete,gemma3:4b,3.3,False,333.3
7,autocomplete,gemma3:4b,3.3,True,2697.8
8,autocomplete,gpt-oss:20b,14.0,False,458.6
9,autocomplete,gpt-oss:20b,14.0,True,2417.7
10,autocomplete,gpt-oss:120b,65.0,False,812.0
11,autocomplete,gpt-oss:120b,65.0,True,3374.0
0,instruct,gemma3:4b,3.3,False,1648.4
1,instruct,gemma3:4b,3.3,True,4329.7
2,instruct,gpt-oss:20b,14.0,False,1336.7
3,instruct,gpt-oss:20b,14.0,True,3624.2


## 6. LaTeX tables (paper)

Generates `benchmark_tables.tex` in this folder. Best value per column is **bold** within baseline and CodeGuard groups.


In [ ]:
def _best_in_group(rows: list[tuple[float, float, float]]) -> tuple[set[int], set[int], set[int]]:
    bleu_vals = [r[0] for r in rows]
    vuln_vals = [r[1] for r in rows]
    pass_vals = [r[2] for r in rows]
    return (
        {i for i, v in enumerate(bleu_vals) if v == max(bleu_vals)},
        {i for i, v in enumerate(vuln_vals) if v == min(vuln_vals)},
        {i for i, v in enumerate(pass_vals) if v == max(pass_vals)},
    )


def _fmt_bleu(value: float, bold: bool = False) -> str:
    text = f"{value:.2f}"
    return f"\\textbf{{{text}}}" if bold else text


def _fmt_pct(value: float, bold: bool = False) -> str:
    text = f"{value:.2f}\\%"
    return f"\\textbf{{{text}}}" if bold else text


def latex_benchmark_table(stat: dict[str, Any], *, caption: str, label: str) -> str:
    baseline_rows = [
        (
            stat[key]["python"]["bleu"],
            stat[key]["python"]["vulnerable_percentage"],
            stat[key]["python"]["pass_rate"],
        )
        for key in SIZE_BASE_MODELS
    ]
    cg_keys = [PREFERRED_CODEGUARD_KEY[m] for m in SIZE_BASE_MODELS]
    cg_rows = [
        (
            stat[key]["python"]["bleu"],
            stat[key]["python"]["vulnerable_percentage"],
            stat[key]["python"]["pass_rate"],
        )
        for key in cg_keys
    ]
    b_bleu, b_vuln, b_pass = _best_in_group(baseline_rows)
    c_bleu, c_vuln, c_pass = _best_in_group(cg_rows)

    lines = [
        r"\begin{table}[h]",
        r"\centering",
        r"\begin{tabular}{|l|c|c|c|}",
        r"\hline",
        r"\textbf{Model} & \textbf{BLEU} $\uparrow$ & \textbf{Vulnerable \%} $\downarrow$ & \textbf{Pass Rate} $\uparrow$ \\",
        r"\hline",
    ]
    for i, model in enumerate(SIZE_BASE_MODELS):
        bleu, vuln, pas = baseline_rows[i]
        lines.append(
            f"{model} & {_fmt_bleu(bleu, i in b_bleu)} & {_fmt_pct(vuln, i in b_vuln)} & {_fmt_pct(pas, i in b_pass)} \\"
        )
    lines.append(r"\hline")
    for i, model in enumerate(cg_keys):
        bleu, vuln, pas = cg_rows[i]
        lines.append(
            f"\\textbf{{{model}}} & {_fmt_bleu(bleu, i in c_bleu)} & {_fmt_pct(vuln, i in c_vuln)} & {_fmt_pct(pas, i in c_pass)} \\"
        )
    lines += [
        r"\hline",
        r"\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        r"\end{table}",
    ]
    return "\n".join(lines)


def print_latex_tables() -> None:
    instruct = latex_benchmark_table(
        load_stat(INSTRUCT_STAT),
        caption="CyberSecEval4 Instruct Benchmark Results",
        label="tab:instruct_benchmark",
    )
    autocomplete = latex_benchmark_table(
        load_stat(AUTOCOMPLETE_STAT),
        caption="CyberSecEval4 Autocomplete Benchmark Results",
        label="tab:autocomplete_benchmark",
    )
    combined = instruct + "\n\n" + autocomplete
    out = EVAL_DIR / "benchmark_tables.tex"
    out.write_text(combined + "\n")
    print(f"Wrote {out}\n")
    print(instruct)
    print()
    print(autocomplete)


print_latex_tables()
